# Phase 6: Probability & Hypothesis Testing (Titanic)

**Objective:** Use probability theory and inferential statistics to mathematically prove the survival dynamics of the Titanic disaster.

In this notebook, we will:
1. Calculate Marginal and Conditional probabilities (e.g., probability of surviving given gender).
2. Apply Bayes' Theorem practically to demonstrate analytical depth.
3. Perform a Chi-Square Test of Independence to statistically prove that socio-economic status (`Pclass`) was a determining factor in survival.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent.parent))
from src.utils.stats import run_chi_square

data_path = Path.cwd().parent.parent / "datasets" / "processed" / "titanic_cleaned.csv"
df = pd.read_csv(data_path)
print(f"Titanic dataset loaded successfully. Shape: {df.shape}")

Titanic dataset loaded successfully. Shape: (889, 11)


### 1. Applied Probability Theory
We often look at basic percentages in EDA, but formalizing them into probability notation for statistical analysis.

- **Marginal Probability $P(A)$:** The probability of an event occurring unconditionally.
- **Conditional Probability $P(A|B)$:** The probability of event A given that event B has occurred.

In [7]:
# 1. Exhaustive Marginal Probabilities (The Baseline of the Ship)
p_survived = df['Survived'].mean()
p_female = (df['Sex'] == 'female').mean()
p_male = (df['Sex'] == 'male').mean()
p_class1 = (df['Pclass'] == 1).mean()
p_class2 = (df['Pclass'] == 2).mean()
p_class3 = (df['Pclass'] == 3).mean()

print("--- MARGINAL PROBABILITIES (Who was on the ship?) ---")
print(f"P(Survived) = {p_survived:.4f}")
print(f"P(Female) = {p_female:.4f}  |  P(Male) = {p_male:.4f}")
print(f"P(Class 1) = {p_class1:.4f} |  P(Class 2) = {p_class2:.4f} |  P(Class 3) = {p_class3:.4f}\n")

--- MARGINAL PROBABILITIES (Who was on the ship?) ---
P(Survived) = 0.3825
P(Female) = 0.3510  |  P(Male) = 0.6490
P(Class 1) = 0.2407 |  P(Class 2) = 0.2070 |  P(Class 3) = 0.5523



In [8]:
# 2. Exhaustive Conditional Probabilities (Who survived?)
p_surv_given_female = df[df['Sex'] == 'female']['Survived'].mean()
p_surv_given_male = df[df['Sex'] == 'male']['Survived'].mean()
p_surv_given_c1 = df[df['Pclass'] == 1]['Survived'].mean()
p_surv_given_c2 = df[df['Pclass'] == 2]['Survived'].mean()
p_surv_given_c3 = df[df['Pclass'] == 3]['Survived'].mean()

print("--- CONDITIONAL PROBABILITIES (Survival rates by demographic) ---")
print(f"P(Survived | Female)  = {p_surv_given_female:.4f}")
print(f"P(Survived | Male)    = {p_surv_given_male:.4f}")
print(f"P(Survived | Class 1) = {p_surv_given_c1:.4f}")
print(f"P(Survived | Class 2) = {p_surv_given_c2:.4f}")
print(f"P(Survived | Class 3) = {p_surv_given_c3:.4f}\n")

--- CONDITIONAL PROBABILITIES (Survival rates by demographic) ---
P(Survived | Female)  = 0.7404
P(Survived | Male)    = 0.1889
P(Survived | Class 1) = 0.6262
P(Survived | Class 2) = 0.4728
P(Survived | Class 3) = 0.2424



### 2. Bayes' Theorem in Practice
Bayes' Theorem allows us to reverse the conditional probabilities. We know the probability of surviving *if* you were a female ($P(Survived|Female)$). 

But what if we pick a survivor at random from the rescue boats? What is the probability that this survivor is female? 
We calculate $P(Female | Survived)$ using Bayes' Theorem:

$$P(Female | Survived) = \frac{P(Survived | Female) \times P(Female)}{P(Survived)}$$

In [9]:
# 3. Exhaustive Bayes' Theorem (Demographics of the Survivors)
# P(Demographic | Survived) = [P(Survived | Demographic) * P(Demographic)] / P(Survived)
p_female_given_surv = (p_surv_given_female * p_female) / p_survived
p_male_given_surv = (p_surv_given_male * p_male) / p_survived
p_c1_given_surv = (p_surv_given_c1 * p_class1) / p_survived
p_c2_given_surv = (p_surv_given_c2 * p_class2) / p_survived
p_c3_given_surv = (p_surv_given_c3 * p_class3) / p_survived

print("--- BAYES' THEOREM (If we pick a random survivor, who are they?) ---")
print(f"P(Female | Survived)  = {p_female_given_surv:.4f}")
print(f"P(Male | Survived)    = {p_male_given_surv:.4f}")
print(f"P(Class 1 | Survived) = {p_c1_given_surv:.4f}")
print(f"P(Class 2 | Survived) = {p_c2_given_surv:.4f}")
print(f"P(Class 3 | Survived) = {p_c3_given_surv:.4f}")

--- BAYES' THEOREM (If we pick a random survivor, who are they?) ---
P(Female | Survived)  = 0.6794
P(Male | Survived)    = 0.3206
P(Class 1 | Survived) = 0.3941
P(Class 2 | Survived) = 0.2559
P(Class 3 | Survived) = 0.3500


### 3. Hypothesis Testing: Chi-Square Test of Independence
In EDA, we saw that 1st Class passengers had a higher survival rate than 3rd Class passengers. We will use a **Chi-Square Test of Independence** to mathematically prove that survival was dependent on passenger class, rather than just random variation.

**Hypotheses:**
- $H_0$: `Pclass` and `Survived` are entirely independent. (Class didn't matter).
- $H_1$: There is a dependent association between `Pclass` and `Survived`.
- **Significance Level ($\alpha$):** 0.05

In [6]:
# Run Chi-Square Test
chi2_result = run_chi_square(df['Pclass'], df['Survived'])

print(f"Test: {chi2_result['test']}")
print(f"Statistic: {chi2_result['statistic']:.2f}")
print(f"Degrees of Freedom: {chi2_result['dof']}")
print("-" * 50)
print(chi2_result['conclusion'])

# Run Chi-Square Test for Sex vs Survived
chi2_sex_result = run_chi_square(df['Sex'], df['Survived'])

print(f"Test: {chi2_sex_result['test']} (Sex vs Survived)")
print(f"Statistic: {chi2_sex_result['statistic']:.2f}")
print(f"Degrees of Freedom: {chi2_sex_result['dof']}")
print("-" * 50)
print(chi2_sex_result['conclusion'])

Test: Chi-Square Test of Independence
Statistic: 100.98
Degrees of Freedom: 2
--------------------------------------------------
Reject Null Hypothesis (p=0.0000 < 0.05): Dependence/Association between the categorical variables is statistically significant.
Test: Chi-Square Test of Independence (Sex vs Survived)
Statistic: 258.43
Degrees of Freedom: 1
--------------------------------------------------
Reject Null Hypothesis (p=0.0000 < 0.05): Dependence/Association between the categorical variables is statistically significant.


### 4. Comprehensive Phase 6 Conclusion (Titanic)

Through an exhaustive combination of probability theory and formal hypothesis testing, we have mathematically solidified the survival dynamics of the Titanic, removing all selective bias.

**1. The "Inversion of Proportions" (Gender & Class):**
Our exhaustive probability calculations revealed a fascinating demographic inversion. 
- **Gender:** Females made up only 35.1% of the ship, but because of their massive 74.0% survival rate, they constituted **67.9% of the survivors** (proven via Bayes' Theorem). 
- **Class:** A similar dynamic occurred with socio-economics. 1st Class was a minority on the ship (~24%), but their 62.6% survival rate meant they out-populated the massive 3rd Class demographic within the lifeboats. 

**2. Absolute Statistical Proof:**
The Chi-Square tests for Independence yielded massive test statistics for both Gender (Stat: 258.43) and Pclass (Stat: 100.98), resulting in p-values of 0.0000. We vehemently reject the null hypotheses; the "women and children first" protocol and socio-economic priority were not random variance, but absolute determinative factors in passenger survival.

**3. Architectural Implications for Modeling:**
- **Baseline Accuracy:** Because overall $P(Survived)$ is ~38.2%, a "dumb" baseline model predicting "Everyone Dies" would automatically achieve ~61.8% accuracy. Our machine learning pipeline must beat this 61.8% floor to be considered statistically useful.
- **Categorical Integrity:** Both `Sex` and `Pclass` are mathematically proven as our highest-signal features. However, passing `Pclass` as integers (1, 2, 3) into linear-based algorithms will confuse the model into treating it as a continuous scale. We must apply strict One-Hot Encoding in our Week 4 preprocessing pipeline.